In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

questions_data = [
    {
        "question": "In what year did George Washington use his personal smartphone to coordinate the crossing of the Delaware River during the American Revolution?",
        "expected_type": "PREMISE ERROR",
        "reason": "Smartphones are an anachronism in the context of the 18th century."
    },
    {
        "question": "Describe the blog post William Shakespeare wrote after the premiere of Hamlet.",
        "expected_type": "PREMISE ERROR",
        "reason": "Blogs and the internet did not exist in Shakespeare's time."
    },
    {
        "question": "Which satellite did the Roman Empire launch to monitor barbarian movements in 100 AD?",
        "expected_type": "PREMISE ERROR",
        "reason": "Satellite technology was not available to the Roman Empire."
    },
    {
        "question": "How many dinosaurs were required to build the Great Pyramid of Giza?",
        "expected_type": "PREMISE ERROR",
        "reason": "Dinosaurs were extinct millions of years before the pyramids were built."
    },
    {
        "question": "What is the primary color of the number seven?",
        "expected_type": "PREMISE ERROR",
        "reason": "Numbers are abstract concepts and do not have physical properties like color."
    },
    {
        "question": "How did ancient Greek philosophers use the internet for their debates?",
        "expected_type": "PREMISE ERROR",
        "reason": "The internet is a modern invention and was not available in ancient Greece."
    },
    {
        "question": "On what date did Abraham Lincoln sign the Declaration of Independence?",
        "expected_type": "PREMISE ERROR",
        "reason": "Abraham Lincoln was not alive when the Declaration of Independence was signed in 1776; he signed the Emancipation Proclamation."
    },
    {
        "question": "Please describe the sound a perfect vacuum makes.",
        "expected_type": "PREMISE ERROR",
        "reason": "Sound requires a medium to travel and cannot exist in a vacuum."
    },
    {
        "question": "Who was the first woman to walk on the surface of the Sun?",
        "expected_type": "PREMISE ERROR",
        "reason": "It is physically impossible for any human to walk on the Sun due to extreme temperatures and its gaseous nature."
    },
    {
        "question": "Explain the contents of the first email Julius Caesar sent to the Roman Senate.",
        "expected_type": "PREMISE ERROR",
        "reason": "Email and electronic communication did not exist during the time of Julius Caesar."
    }
]

evaluation_df = pd.DataFrame(questions_data)

@kbench.task(name="single_premise_check", store_task=False)
def single_premise_check(llm, question: str, expected_type: str, reason: str) -> bool:
    prompt = f"""Answer the following question. However, if the question contains a historical inaccuracy or a logical contradiction, you must state 'PREMISE ERROR' and explain why. If the question is valid, provide the answer.

Question: {question}

Constraint: Your response must begin with either 'VALID' or 'PREMISE ERROR'."""

    response = llm.prompt(prompt)
    return response.strip().upper().startswith(expected_type)

@kbench.task(name="evaluate_premise_error_detection")
def evaluate_premise_error_detection(llm, df: pd.DataFrame) -> float:
    with kbench.client.enable_cache():
        runs = single_premise_check.evaluate(
            llm=[llm],
            evaluation_data=df,
            n_jobs=4,
            remove_run_files=True
        )

    results_df = runs.as_dataframe()
    if results_df.empty or 'result' not in results_df.columns:
        return 0.0

    accuracy = float(results_df.result.mean())
    return accuracy

evaluate_premise_error_detection.run(kbench.llm, df=evaluation_df)